## Local practical overview

In this hands-on practical, you will retrieve a PRIDE `.raw` file, convert it to `mzML`, prepare spectra for **InstaNovo**, generate *de novo* peptide-sequence predictions, annotate spectra, and assess prediction confidence and ion coverage. An optional final section calibrates confidence, controls FDR, and performs retained-PSM QC with **winnow**.

This local version installs the platform-compatible PyTorch build and runs from the directory containing the notebook. It can use CPU or Apple Silicon MPS where supported; the prediction step will be slower than the CUDA-enabled Colab version.

*Notebook created by Yun Chiang and Jeroen van Goey.*

# *De novo* sequencing: from a PRIDE `.raw` file to annotated spectra

This is a 45-minute practical that walks you through the following:

1. **Fetch** a Thermo `.raw` file from the PRIDE FTP archive
2. **Convert** `.raw` → `mzML` with **ThermoRawFileParser**
3. **Parse** the `mzML` into a `parquet` for **InstaNovo** CLI and a `mgf` for their Hugging Face space
4. ***De Novo*** predict the spectra with **InstaNovo**
5. **Inspect** predictions and **annotate** spectra with its predicted b/y ions using **spectrum_utils**
6. **Score** predictions with label-free quality **metrics** (confidence & ion coverage)
7. *(optional, advanced)* **Calibrate** confidence and control **FDR** with **winnow**, then **QC** retained PSMs with Winnow metadata

> **Before you start:** open this notebook in a local Jupyter or VS Code environment and select the Python kernel you want to use.

>The working directory is the directory from which your notebook kernel starts. Keep the notebook and its generated files in a writable project directory.

## Step 0 — Local setup (run once)
The setup installs a platform-compatible PyTorch and InstaNovo environment. If packages are installed, restart the notebook kernel once before continuing.

In [ ]:
#@title Local pip install cell
# Safe to re-run: if InstaNovo already imports, the whole setup is skipped.
try:
    import instanovo  # noqa: F401
    print("InstaNovo already installed - skipping setup.")
except ImportError:
    # Install the platform-compatible PyTorch build (CPU/MPS); do not request NVIDIA wheels.
    !pip install -q biopython "instanovo>=1.2.2" pyteomics psims lxml polars pyarrow spectrum_utils ipywidgets
    print("Local installation complete. Restart the kernel once, then run the next cell.")

In [ ]:
#@title Check the local compute device
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())
# Pin down exactly which InstaNovo / InstaNovo+ versions we're running (helps debugging)
!instanovo version

## Step 1 — Fetch a .raw file from PRIDE

Every PRIDE project has a predictable FTP path:

```
https://ftp.pride.ebi.ac.uk/pride/data/archive/<year>/<month>/<accession>/
```
We are going to download a file from [PXD052635](https://www.ebi.ac.uk/pride/archive/projects/PXD052635):
```
20211027_EXPL2_nLC3_MEM_collab_77min_DDA_Ancient_enamel_1963.raw
```

In [ ]:
import os, requests
from tqdm.auto import tqdm

ACCESSION = "PXD052635"
YEAR, MONTH = "2025", "07"
RAW_NAME = "20211027_EXPL2_nLC3_MEM_collab_77min_DDA_Ancient_enamel_1963.raw"
URL = f"https://ftp.pride.ebi.ac.uk/pride/data/archive/{YEAR}/{MONTH}/{ACCESSION}/{RAW_NAME}"
RAW_PATH = RAW_NAME

if not os.path.exists(RAW_PATH):
    print("Downloading", URL)
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total_bytes = int(r.headers.get("Content-Length", 0))
        with open(RAW_PATH, "wb") as f, tqdm(total=total_bytes, unit="B", unit_scale=True, desc=RAW_NAME) as bar:
            for chunk in r.iter_content(1 << 20):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
print("RAW size (MB):", round(os.path.getsize(RAW_PATH) / 1e6, 1))

## Step 2 — Convert Thermo .raw → mzML
`.raw` is Thermo's vendor specific (closed) format. **[ThermoRawFileParser](https://github.com/compomics/ThermoRawFileParser)** is an open-source parser that runs on all platforms to convert `.raw` to open formats: `mzML`/`MGF`. We are using `v.2.0.0-dev` without `Mono` dependency.

The `-f` flag picks the **output format**:

| `-f` | format |
|------|--------|
| 0 | MGF |
| 1 | mzML (plain) |
| 2 | **indexed mzML** — *we use this* |
| 3 | Parquet |
| 4 | none (metadata only) |

We choose **`2` (indexed mzML)**: the embedded index lets a reader jump straight to any spectrum without scanning the whole file, so InstaNovo's `SpectrumDataFrame` loads it quickly. (Plain mzML in option 1 has the same data but no index.)

In [ ]:
%%bash
set -euo pipefail
# Download the self-contained parser for the host operating system.
if [ -f ThermoRawFileParser/ThermoRawFileParser ]; then
    echo "ThermoRawFileParser already present - skipping download."
    exit 0
fi
case "$(uname -s)-$(uname -m)" in
  Darwin-arm64) ASSET=ThermoRawFileParser-v.2.0.0-dev-osx-arm64.zip ;;
  Darwin-x86_64) ASSET=ThermoRawFileParser-v.2.0.0-dev-osx.zip ;;
  Linux-*) ASSET=ThermoRawFileParser-v.2.0.0-dev-linux.zip ;;
  *) echo "Unsupported platform: $(uname -s)-$(uname -m)" >&2; exit 1 ;;
esac
curl -fsSL -O "https://github.com/CompOmics/ThermoRawFileParser/releases/download/v.2.0.0-dev/${ASSET}"
unzip -q "${ASSET}" -d ThermoRawFileParser
rm "${ASSET}"
chmod +x ThermoRawFileParser/ThermoRawFileParser
echo "ThermoRawFileParser downloaded for $(uname -s)-$(uname -m)."

In [ ]:
import os

# Reuse RAW_PATH from Step 1 so the filename lives in exactly one place.
# ThermoRawFileParser writes the .mzML next to the input with the same base name.
MZML_PATH = RAW_PATH.replace(".raw", ".mzML")
fmt = 2  # 0=MGF, 1=mzML (plain), 2=indexed mzML

# Skip the conversion if the .mzML already exists.
if os.path.exists(MZML_PATH):
    print(f"{MZML_PATH} already exists - skipping conversion.")
else:
    !ThermoRawFileParser/ThermoRawFileParser -i "{RAW_PATH}" -f "{fmt}" -z

## Step 3 — Parse the mzML
We are going to see what's inside a `mzML` by using InstaNovo's `SpectrumDataFrame`. `.parquet` is like a `.csv` but smaller to store and faster for a computer to process.

In [ ]:
import pandas as pd
from instanovo.utils.data_handler import SpectrumDataFrame

sdf1 = SpectrumDataFrame.load(MZML_PATH, lazy=False, is_annotated=False)
df = sdf1.to_pandas()
print(f"Parsed {len(df)} spectra | columns: {list(df.columns)}")
df.head()  # show a preview, not all rows (the mz/intensity arrays are large)

For this workshop we select a **subset of 100 spectra** to keep runtimes short.

Rather than the first scans — which elute early and are mostly low quality — we take **charge 2–3** precursors as a **mix of the 250 most intense + 250 random** spectra. The most-intense alone are dominated by a few very abundant peptides (lots of duplicates); adding a random half gives **diverse peptides and a spread of confidence** (some easy, some hard) — useful for the metrics and FDR steps later.

In [ ]:
import numpy as np

# The first scans elute early and are mostly low quality. Instead, build a
# 500-spectrum subset with a MIX of quality, for diverse peptides AND a spread
# of confidence: the most-intense (top-TIC) spectra alone are dominated by a
# few very abundant peptides (lots of duplicates). Keep charge 2-3 (best for
# de novo); take 250 most intense + 250 random.
dfq = df[df["precursor_charge"].isin([2, 3])].copy()
dfq["_tic"] = dfq["intensity_array"].map(lambda a: float(np.sum(a)))
top  = dfq.sort_values("_tic", ascending=False).head(250)
rand = dfq.drop(top.index).sample(n=250, random_state=0)   # random_state -> reproducible
df1  = (pd.concat([top, rand]).drop(columns="_tic")
          .sample(frac=1, random_state=0)                 # shuffle so it isn't quality-sorted
          .reset_index(drop=True))
print(f"selected {len(df1)} spectra: 250 most intense + 250 random (charge 2-3) | median peaks/spectrum: {int(df1['mz_array'].map(len).median())}")

In [ ]:
from pyteomics import mgf
import numpy as np

def df_to_mgf(df, out_path):
    spectra = []
    for _, row in df.iterrows():
        params = {
            "title":       f"{row['experiment_name']}:scan:{int(row['scan_number'])}",
            "pepmass":     float(row["precursor_mz"]),
            "charge":      f"{int(row['precursor_charge'])}+",
            "rtinseconds": float(row["retention_time"]) * 60.0,
            "scans":       int(row["scan_number"]),
        }
        seq = row.get("sequence")
        if seq is not None and str(seq).strip() not in ("", "nan", "None"):
            params["seq"] = str(seq)

        spectra.append({
            "m/z array":       np.asarray(row["mz_array"],        dtype=float),
            "intensity array": np.asarray(row["intensity_array"], dtype=float),
            "params":          params,
        })

    mgf.write(spectra, out_path, file_mode="w")
    return f"Converted dataframe to MGF file: {out_path}"

We then write our subset to both `.parquet` (for the CLI) and `.mgf` (for the Hugging Face space).

In [ ]:
df1["spectrum_id"] = df1["experiment_name"].astype(str) + ":" + df1["scan_number"].astype(str)
df1.to_parquet("subset_500.parquet")
print("Converted dataframe to parquet file: subset_500.parquet")
print(df_to_mgf(df1, "subset_500.mgf"))

## Step 4 — Run InstaNovo for *de novo*
**[InstaNovo](https://github.com/instadeepai/InstaNovo)** is a transformer model that is well-annotated and easy to run. We run it to get a peptide sequence per spectrum, written to `preds.csv` (columns include `predictions`, the sequence confidence `log_probs`, per-residue `token_log_probs`, and beam-search columns).
>
- option A: For Google account users, you may run it here using the CLI (command line interface).
- option B: Go to InstaNovo [Hugging Face space](https://huggingface.co/spaces/InstaDeepAI/InstaNovo) and run the `subset_500.mgf` we made in Step 3 online. <br />(note **not the `.mzML`** since it may be too big)

> InstaNovo also ships **InstaNovo+**, a diffusion model that *refines* the transformer output (`instanovo predict ... --with-refinement`). We skip it here to keep the workshop fast; see the [InstaNovo getting-started notebook](https://github.com/instadeepai/InstaNovo/blob/main/notebooks/getting_started_with_instanovo.ipynb) to try it.

In [ ]:
!instanovo predict --help

In [ ]:
# Run the transformer model for de novo sequencing.
# (--no-refinement: skip the InstaNovo+ diffusion step to keep the workshop fast.)
!instanovo predict --data-path subset_500.parquet --output-path preds.csv --denovo --no-refinement

## Step 5 — Inspect predictions and annotate spectra
We will pick *de novo* predictions from a list of top and bottom 10 spectra (sorted by model confidence), and visualise them using mirrored plots to see if observed b/y ions matched predicted ones. Generally, higher the ion coverage (observed/predicted ions), the better.

This is *real* data with **no ground-truth peptide labels**, so we can't compute recall/precision — we use confidence and b/y ion coverage as quality proxies instead (quantified in Step 6).

In [ ]:
import matplotlib.pyplot as plt
import spectrum_utils.spectrum as sus
import spectrum_utils.plot as sup
from spectrum_utils import proforma
from spectrum_utils.fragment_annotation import get_theoretical_fragments
%matplotlib inline

In [ ]:
PARQUET = "subset_500.parquet"
#change fragment ion tolerance here
TOL, TOL_MODE = 10, "ppm"

preds   = pd.read_csv("preds.csv")
spectra = pd.read_parquet(PARQUET)

seq_col  = next((c for c in ["predictions","prediction","sequence","peptide"] if c in preds.columns), preds.columns[0])
conf_col = next((c for c in ["log_probs","log_probabilities","confidence"] if c in preds.columns), None)
if conf_col is not None:
    preds["model_confidence"] = np.clip(np.exp(preds[conf_col]), 0, 1)

cand = preds.sort_values(conf_col, ascending=False) if conf_col else preds
cols = ["scan_number", seq_col, conf_col, "model_confidence"]

print("==TOP 10 (most confident)==")
print(cand[cols].head(10).to_string(index=False))

print("\n==BOTTOM 10 (least confident)==")
print(cand[cols].tail(10).to_string(index=False))

In [ ]:
def show_mirror(scan_number=None):
    """Pass a scan_number from the list above; None = highest confidence."""
    if scan_number is None:
        row = cand.iloc[0]
    else:
        sel = cand[cand["scan_number"] == scan_number]
        if sel.empty:
            print(f"scan {scan_number} not found in predictions - pick one from the lists above.")
            return
        row = sel.iloc[0]
    peptide, scan = str(row[seq_col]), int(row["scan_number"])
    conf = float(row["model_confidence"]) if "model_confidence" in row else None
    sp = spectra.loc[spectra["scan_number"] == scan].iloc[0]
    pmz, z = float(sp["precursor_mz"]), int(sp["precursor_charge"])

    obs = sus.MsmsSpectrum(str(scan), pmz, z,
                           np.asarray(sp["mz_array"], float), np.asarray(sp["intensity_array"], float))
    obs = (obs.set_mz_range(100, 1500).remove_precursor_peak(TOL, TOL_MODE)
              .filter_intensity(0.01, 50).scale_intensity("root")
              .annotate_proforma(peptide, TOL, TOL_MODE, ion_types="by"))

    proteoform = proforma.parse(peptide)[0]
    frags = get_theoretical_fragments(proteoform, ion_types="by", max_charge=1)
    frag_mz = np.array([mz for _, mz in frags])
    theo = sus.MsmsSpectrum(f"{peptide} (theoretical)", pmz, z,
                            frag_mz, np.full(len(frag_mz), obs.intensity.max()))
    theo = theo.annotate_proforma(peptide, TOL, TOL_MODE, ion_types="by")

    fig, ax = plt.subplots(figsize=(12, 6))
    sup.mirror(obs, theo, ax=ax)

    # Restyle the lower (theoretical, negative-y) half of the mirror plot as
    # dashed lines. This pokes at matplotlib artists directly, so it may need
    # updating if spectrum_utils / matplotlib change how they draw peaks.
    for line in ax.lines:
        yd = line.get_ydata()
        if len(yd) and np.min(yd) < -1e-9:
            line.set_linestyle("--")
    for coll in ax.collections:
        segs = coll.get_segments()
        if segs and min(s[:, 1].min() for s in segs) < -1e-9:
            coll.set_linestyle("--")

    conf_str = f"  |  confidence {conf:.2f}" if conf is not None else ""
    ax.set_title(f"scan {scan}  |  {peptide}{conf_str}   (top: observed   bottom: theoretical)")
    plt.tight_layout(); plt.show()

# highest-confidence prediction
show_mirror()

In [ ]:
# show_mirror() above used the most-confident prediction. You can pass any
# scan_number from the TOP/BOTTOM lists above. Scan numbers depend on the
# (random) subset, so here we pick the LEAST-confident one to contrast.
show_mirror(int(cand.iloc[-1]["scan_number"]))

Show the first few predictions.

In [ ]:
preds.head()

## Step 6 — Quality metrics without ground truth
This is real data with **no known peptides**, so we can't compute recall/precision. Instead we use two **label-free** signals:
- **model confidence** = `exp(log_probs)` of the prediction — how sure the model is.
- **ion coverage** = fraction of the predicted peptide's theoretical **b/y** ions actually observed in the spectrum — how well the prediction explains the data.

A trustworthy identification has **both** high confidence *and* high ion coverage.
> Our Step 3 subset deliberately mixes high-quality and random spectra, so you'll see a **spread**: confident, well-covered hits alongside poor ones. Watch for the disagreements — **high confidence with low coverage is a red flag**, and is exactly what confidence-calibration / FDR-control tools (e.g. [winnow](https://github.com/instadeepai/winnow), see later) are built to catch.

In [ ]:
#@title Confidence & ion coverage (reuses spectrum_utils imports from Step 5)
preds   = pd.read_csv("preds.csv")
spectra = pd.read_parquet("subset_500.parquet")
TOL = 10  # ppm, fragment match tolerance

preds["model_confidence"] = np.clip(np.exp(preds["log_probs"]), 0, 1)

def ion_coverage(peptide, mz_array, tol_ppm=TOL):
    """Fraction of theoretical singly-charged b/y ions observed in the spectrum."""
    try:
        prot = proforma.parse(str(peptide))[0]
        frags = get_theoretical_fragments(prot, ion_types="by", max_charge=1)
    except Exception:
        return np.nan
    obs = np.asarray(mz_array, float)
    if not frags or obs.size == 0:
        return np.nan
    matched = sum(np.min(np.abs(obs - t)) / t * 1e6 <= tol_ppm for _, t in frags)
    return matched / len(frags)

mz_by_scan = {int(r["scan_number"]): r["mz_array"] for _, r in spectra.iterrows()}
preds["ion_coverage"] = [ion_coverage(p, mz_by_scan.get(int(s)))
                         for p, s in zip(preds["predictions"], preds["scan_number"])]

print("model confidence : mean %.2f  median %.2f" % (preds["model_confidence"].mean(), preds["model_confidence"].median()))
print("ion coverage     : mean %.2f  median %.2f" % (preds["ion_coverage"].mean(), preds["ion_coverage"].median()))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(preds["model_confidence"], bins=20); ax[0].set(title="Model confidence", xlabel="exp(log_probs)", ylabel="# spectra")
ax[1].hist(preds["ion_coverage"].dropna(), bins=20); ax[1].set(title="Ion coverage (b/y)", xlabel="fraction observed")
ax[2].scatter(preds["model_confidence"], preds["ion_coverage"], alpha=0.6); ax[2].set(title="Confidence vs coverage", xlabel="model confidence", ylabel="ion coverage")
plt.tight_layout(); plt.show()

In [ ]:
#@title Per-residue confidence
# InstaNovo reports a per-residue confidence (token log-probability); low-confidence
# residues are where the model is unsure (often where errors or isobaric swaps happen).
import ast

def show_residue_confidence(scan_number):
    row = preds.loc[preds["scan_number"] == scan_number]
    if row.empty:
        print(f"scan {scan_number} not found - pick one from the lists above.")
        return
    row = row.iloc[0]
    tokens = [t.strip() for t in str(row["predictions_tokenised"]).split(",")]
    tlp = np.asarray(ast.literal_eval(row["token_log_probs"]), float)
    conf = np.exp(tlp[:len(tokens)])

    _, ax = plt.subplots(figsize=(max(6, 0.35 * len(tokens)), 3.3)) # slightly closer stems, more vertical room
    markerline, stemlines, _ = ax.stem(range(len(tokens)), conf, basefmt=" ")
    # Bold marker and stem lines
    plt.setp(markerline, markersize=7)
    plt.setp(stemlines, linewidth=2)
    # Make stems closer
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    # Add breathing room at the top
    y_max = min(1.05, max(1, conf.max() + 0.08))
    ax.set_ylim(-0.05, y_max)
    ax.set_ylabel("Residue confidence")
    ax.set_title(f"Scan {scan_number}: {''.join(tokens)}")
    plt.tight_layout(); plt.show()

# most confident prediction
show_residue_confidence(int(preds.sort_values("model_confidence", ascending=False).iloc[0]["scan_number"]))

In [ ]:
# least confident prediction
show_residue_confidence(int(preds.sort_values("model_confidence", ascending=False).iloc[-1]["scan_number"]))

### Biological sanity check — does the top hit make sense?
Interpret the most confident predictions in the context of [PXD052635](https://www.ebi.ac.uk/pride/archive/projects/PXD052635), an ancient-enamel dataset. The observed peptide composition should be evaluated against the sample context and expected diagenetic modification patterns.

In [ ]:
#@title Most frequent high-confidence predictions
confident = preds[preds["model_confidence"] >= 0.9]
print(f"{len(confident)} / {len(preds)} predictions with confidence >= 0.9; most frequent peptides:\n")
print(confident["predictions"].value_counts().head(10).to_string())

So let's look up what it is: we'll query **UniProt's peptide search** (exact match across UniProt — no BLAST needed) and report the hits. Recovering a known protein's peptide **with no database used during sequencing** is a strong sign the *de novo* pipeline is working. *(Needs internet.)*

In [ ]:
#@title Which protein is it? (UniProt peptide search - no BLAST needed)
import urllib.request, urllib.error, json, time

peptide = confident["predictions"].value_counts().idxmax()   # most frequent confident peptide
print("querying UniProt peptide search for:", peptide)

# 1) submit an async exact-peptide search, restricted to Bos taurus (taxId 9913)
r = urllib.request.urlopen(urllib.request.Request(
    "https://peptidesearch.uniprot.org/asyncrest",
    data=f"peps={peptide}&taxIds=9913&lEQi=off&spOnly=off".encode(),
    headers={"Content-Type": "application/x-www-form-urlencoded"}, method="POST"), timeout=60)
job = r.headers["Location"].replace("http://", "https://")

# 2) poll until the job returns results
accs = None
for _ in range(40):
    try:
        resp = urllib.request.urlopen(job, timeout=60)
        if resp.status == 200:
            accs = resp.read().decode().strip().split(","); break
    except urllib.error.HTTPError:
        pass
    time.sleep(2)

# 3) fetch the top hit's name + sequence and check where the peptide sits
if not accs:
    print("No UniProt match (or the service timed out).")
else:
    acc = accs[0]
    j = json.load(urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=60))
    name = j["proteinDescription"]["recommendedName"]["fullName"]["value"]
    org  = j["organism"]["scientificName"]
    seq  = j["sequence"]["value"]
    sig  = [f for f in j.get("features", []) if f["type"] == "Signal"]
    sig_end = sig[0]["location"]["end"]["value"] if sig else 0
    pos  = seq.find(peptide) + 1
    print(f"\n{peptide}  ->  {acc}: {name} ({org})")
    print(f"  precursor residues {pos}-{pos + len(peptide) - 1}; ends in '{peptide[-1]}' -> tryptic C-terminus: {peptide[-1] in 'KR'}")
    if sig_end:
        where = "N-terminus of the mature protein" if pos == sig_end + 1 else "internal peptide"
        print(f"  signal peptide ends at residue {sig_end} -> mature protein starts at {sig_end + 1}: this is the {where}")
    print(f"  {len(accs)} UniProt match(es) (incl. beta-LG variants): {', '.join(accs)}")

So **`LIVTQTMK`** is the N-terminal tryptic peptide of mature **bovine β-lactoglobulin**

## Step 7 Calibrate confidence & control FDR with winnow
The model's `log_probs` confidence isn't *calibrated* — a score of 0.9 doesn't literally mean "90% likely correct". [**Winnow**](https://github.com/instadeepai/winnow) recalibrates *de novo* confidence with a **pretrained general model** and estimates a **false discovery rate (FDR)** *without ground-truth labels*, so you can keep only PSMs below a chosen FDR.

**Heads-up:**
Needs **internet**: it downloads a pretrained calibrator from Hugging Face and queries the public **[Koina](https://koina.wilhelmlab.org)** server for fragment-ion predictions.

In [ ]:
#@title Install winnow (confidence calibration & FDR control)
!pip install -q "winnow-fdr>=2.0.0"

In [ ]:
!winnow predict --help

The pretrained calibrator's Prosit fragment model needs a per-spectrum collision energy and fragmentation type, which aren't in the mzML metadata.
For this run (Q Exactive HF, HCD) the normalized collision energy (NCE) is ~30.
Set these parameters to your acquisition's normalized collision energy for the most accurate fragment-match features.

We also need to increase the number of training samples to train observed retention time to indexed retention time regressors, since our most confident predictions are often duplicates.

In [ ]:
# We set the FDR threshold to 1.0 to return all PSMs
!winnow predict dataset.spectrum_path_or_directory=subset_500.parquet \
                dataset.predictions_path=preds.csv koina.input_constants.collision_energies=30 \
                koina.input_constants.fragmentation_types=HCD \
                output_folder=winnow_results \
                fdr_control.fdr_threshold=1.0 \
                +calibrator.irt_calibration.train_fraction=0.3

In [ ]:
_HEAVY_COLS = {
    "mz_array", "intensity_array", "theoretical_mz", "theoretical_intensity",
    "theoretical_annotation", "ion_matches", "ion_match_intensity", "token_log_probs",
}
metadata = pd.read_csv(
    "winnow_results/metadata.csv",
    usecols=lambda c: c not in _HEAVY_COLS,
)
preds_and_fdr_metrics = pd.read_csv("winnow_results/preds_and_fdr_metrics.csv")
winnow_results = metadata.merge(preds_and_fdr_metrics, on="spectrum_id")

print(winnow_results["calibrated_confidence"].describe())

In [ ]:
print("PSMs passing 5% FDR:", int((preds_and_fdr_metrics["psm_q_value"] <= 0.05).sum()))
print("PSMs passing 10% FDR:", int((preds_and_fdr_metrics["psm_q_value"] <= 0.1).sum()))
print("PSMs passing 25% FDR:", int((preds_and_fdr_metrics["psm_q_value"] <= 0.25).sum()))

In [ ]:
print("==TOP 10 (most confident after calibration)==")
print(preds_and_fdr_metrics.sort_values("calibrated_confidence", ascending=False).head(10).to_string(index=False))

print("\n==BOTTOM 10 (least confident after calibration)==")
print(preds_and_fdr_metrics.sort_values("calibrated_confidence", ascending=False).tail(10).to_string(index=False))

In [ ]:
fdr_curve = preds_and_fdr_metrics.sort_values("calibrated_confidence")
plt.figure(figsize=(7, 4))
plt.plot(fdr_curve["calibrated_confidence"], fdr_curve["psm_fdr"], marker=".")
plt.xlabel("Calibrated confidence")
plt.ylabel("Estimated PSM FDR")
plt.title("Winnow: estimated FDR vs calibrated confidence"); plt.grid(True); plt.show()

### Reading the FDR curve
The curve is **monotonic**: higher calibrated confidence → lower estimated FDR. To control the false discovery rate you pick a target and keep only PSMs **above the matching confidence cutoff**:
- **5% FDR** needs calibrated confidence ≈ **0.87** here → about **44 / 100** PSMs pass.
- relaxing to **10%** drops the cutoff (≈ 0.70) and keeps more; **25%** keeps almost everything.

This is the **precision/recall trade-off made explicit**: a stricter FDR keeps fewer but higher-quality identifications. The key win is that it works **without ground-truth labels** — after calibration the confidence behaves like a real probability of being correct, so "5% FDR" means roughly 5% of the *kept* PSMs are expected to be wrong. For downstream analysis you'd report only the FDR-filtered subset (e.g. the ~44 PSMs at 5% FDR), not all 100 raw predictions.

> ⚠️ The estimate is only as trustworthy as its inputs: a **generic** pretrained calibrator, an **approximate** collision energy, and the non-parametric method's assumption that the calibrated scores really are well-calibrated. Treat the exact cutoffs as indicative, and re-calibrate on your own instrument/data for production use.

In [ ]:
winnow_results.head()

### Follow-on: QC retained PSMs with Winnow metadata

FDR filtering gives you a list to report, but practitioners still inspect individual PSMs before publication or quantitation. Winnow's `metadata.csv` holds _calibration features_ that mirror standard database-search QC (precursor mass error, spectral similarity, retention time) plus de novo-specific signals (beam ambiguity, per-residue confidence).

Step 6 used InstaNovo's raw `model_confidence` and a hand-rolled `ion_coverage` (counting matched b/y peaks). Here we use Winnow's computed features especially `spectral_angle`, `mass_error_da`, and `margin`, which are close to what search engines report and what the calibrator actually learned from.

We'll work on the **5% FDR retained subset** (~47 PSMs on this run), not all raw predictions.

In [ ]:
FDR_THRESHOLD = 0.05

if "ion_coverage" not in winnow_results.columns and "ion_coverage" in preds.columns:
    winnow_results = winnow_results.merge(
        preds[["scan_number", "ion_coverage"]], on="scan_number", how="left"
    )

retained = winnow_results[winnow_results["psm_q_value"] <= FDR_THRESHOLD].copy()
rejected = winnow_results[winnow_results["psm_q_value"] > FDR_THRESHOLD].copy()
print(f"Retained at {FDR_THRESHOLD:.0%} FDR: {len(retained)} / {len(winnow_results)} PSMs")

QC_METRICS = [
    "calibrated_confidence", "spectral_angle", "complementary_ion_count",
    "max_ion_gap", "margin", "edit_distance", "min_token_probability", "irt_error",
]

def _qc_summary(df, label):
    row = {"group": label, "n": len(df)}
    for col in QC_METRICS:
        row[col] = df[col].median()
    row["|mass_error_da|"] = df["mass_error_da"].abs().median()
    row["|delta_mass_ppm|"] = df["delta_mass_ppm"].abs().median()
    return row

summary = pd.DataFrame([
    _qc_summary(retained, "retained (q≤5%)"),
    _qc_summary(rejected, "rejected"),
]).set_index("group")
display(summary.round(3))

#### Fragment-ion evidence
Winnow predicts a theoretical spectrum using [Prosit](https://koina.wilhelmlab.org/docs#post-/Prosit_2025_intensity_22PTM/infer) (Koina, CE≈30 / HCD in our command) and compares it to the observed spectrum:
- **`spectral_angle`** — normalized spectral similarity
- **`complementary_ion_count`** / **`max_ion_gap`** — how complete the b/y ion ladder is

High calibrated confidence with low `spectral_angle` is a similar red flag as Step 6's "high confidence, low coverage"

In [ ]:
pass_mask = winnow_results["psm_q_value"] <= FDR_THRESHOLD
colors = np.where(pass_mask, "C0", "C1")

import matplotlib.lines as mlines

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(retained["spectral_angle"], bins=20, color="C0", alpha=0.85)
ax[0].set(title="Spectral angle (retained)", xlabel="Spectral angle", ylabel="# PSMs")

# Scatter: Calibrated confidence vs spectral angle
scatter1 = ax[1].scatter(
    winnow_results["calibrated_confidence"], winnow_results["spectral_angle"],
    c=colors, alpha=0.65, s=35,
)
ax[1].axhline(0.4, color="gray", ls="--", lw=1, alpha=0.7)
ax[1].set(title="Confidence vs spectral angle", xlabel="Calibrated confidence", ylabel="Spectral angle")

legend_handles = [
    mlines.Line2D([], [], color="C0", marker='o', linestyle='None', markersize=8, label=f'Retained (q≤{FDR_THRESHOLD:.0%})'),
    mlines.Line2D([], [], color="C1", marker='o', linestyle='None', markersize=8, label='Rejected'),
]
ax[1].legend(handles=legend_handles)

# Scatter: Ion ladder (retained & rejected, colored by pass/fail)
scatter2 = ax[2].scatter(
    winnow_results["complementary_ion_count"], winnow_results["max_ion_gap"],
    c=colors, alpha=0.65, s=35,
)
ax[2].set(title="Ion ladder completeness", xlabel="Complementary ion count", ylabel="Maximum ion gap")
# Legend for this plot as well
ax[2].legend(handles=legend_handles)

plt.tight_layout(); plt.show()

if "ion_coverage" in winnow_results.columns:
    fig, ax = plt.subplots(figsize=(5, 4))
    sc = ax.scatter(winnow_results["ion_coverage"], winnow_results["spectral_angle"], c=colors, alpha=0.65, s=35)
    ax.set(xlabel="Ion coverage", ylabel="Spectral angle", title="Spectral evidence")
    # Add legend for the scatter plot color meanings
    legend_handles2 = [
        mlines.Line2D([], [], color="C0", marker='o', linestyle='None', markersize=8, label=f'Retained (q≤{FDR_THRESHOLD:.0%})'),
        mlines.Line2D([], [], color="C1", marker='o', linestyle='None', markersize=8, label='Rejected'),
    ]
    ax.legend(handles=legend_handles2)
    plt.tight_layout(); plt.show()

#### Precursor mass accuracy
Winnow reports isotope-corrected signed neutral-mass error in Daltons.

Confident PSMs should cluster near 0 Da. Errors of even a few Da usually mean wrong sequence, modification or isotope/charge assignment, regardless of model confidence.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].hist(retained["mass_error_da"], bins=20, color="C0", alpha=0.85)
ax[0].set(title="Mass error (retained)", xlabel="Precursor mass error (Da)", ylabel="# PSMs")

ax[1].scatter(
    winnow_results["calibrated_confidence"], winnow_results["mass_error_da"],
    c=colors, alpha=0.65, s=35,
)
ax[1].axhline(0, color="gray", lw=0.8)
ax[1].set(title="Confidence vs mass error (Da)", xlabel="Calibrated confidence", ylabel="Precursor mass error (Da)")
plt.tight_layout(); plt.show()

#### De novo ambiguity
When InstaNovo exports multiple beam candidates, among other features, Winnow computes:
- **`margin`** — probability gap between top-1 and top-2 sequences
- **`min_token_probability`** — weakest residue in the top prediction

Low margin on a retained PSM means several similar sequences competed.

A negative margin results from InstaNovo's internal beam ranking system: sort by precursor mass fit first, then secondly by confidence.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
bins = np.linspace(0, 1, 21)
ax[0].hist(rejected["margin"], bins=bins, alpha=0.5, label="Rejected", color="C1")
ax[0].hist(retained["margin"], bins=bins, alpha=0.5, label="Retained", color="C0")
ax[0].legend(); ax[0].set(title="Beam margin", xlabel="Margin", ylabel="# PSMs")

ax[1].scatter(
    winnow_results["calibrated_confidence"], winnow_results["margin"],
    c=colors, alpha=0.65, s=35,
)
ax[1].set(title="Confidence vs beam margin", xlabel="Calibrated confidence", ylabel="Margin")

ax[2].hist(retained["min_token_probability"], bins=20, color="C0", alpha=0.85)
ax[2].set(title="Weakest residue (retained)", xlabel="Minimum token probability", ylabel="# PSMs")
plt.tight_layout(); plt.show()

#### Retention-time consistency
Winnow fits a per-run RT→iRT regressor on high-confidence PSMs (no database labels needed). Large **`irt_error`** on a retained PSM can flag a wrong sequence or modification.

In [ ]:
irt_ok = ~winnow_results["is_missing_irt_error"].astype(bool)
irt_df = winnow_results.loc[irt_ok]
irt_ret = retained.loc[~retained["is_missing_irt_error"].astype(bool)]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
lim = [irt_df["iRT"].min(), irt_df["iRT"].max()]
ax[0].scatter(
    irt_df["iRT"], irt_df["predicted iRT"],
    c=np.where(irt_df["psm_q_value"] <= FDR_THRESHOLD, "C0", "C1"), alpha=0.65, s=35,
)
ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set(title="Observed vs predicted iRT", xlabel="Prosit-predicted iRT", ylabel="Winnow-predicted iRT")

ax[1].hist(irt_ret["irt_error"], bins=20, color="C0", alpha=0.85)
ax[1].set(title="iRT error (retained)", xlabel="iRT error", ylabel="# PSMs")
plt.tight_layout(); plt.show()

#### Triage and next steps
Even among FDR-retained PSMs, simple heuristics catch cases worth manual review. Feel free to change these flag values and examine retained PSMs.

In [ ]:
flags = (
    (retained["spectral_angle"] < 0.4)
    | (retained["mass_error_da"].abs() > 0.02)
    | (retained["irt_error"] > 10)
)
review_cols = [
    "scan_number", "prediction", "calibrated_confidence", "psm_q_value",
    "spectral_angle", "ion_coverage", "mass_error_da", "margin",
    "min_token_probability", "std_token_probability", "irt_error",
]
review = retained.loc[flags, review_cols].sort_values("calibrated_confidence", ascending=False)
print(f"Retained PSMs flagged for manual review: {len(review)} / {len(retained)}")
display(review.round(4))